In [1]:
# Direct imports: torch, transformers, peft trl, datasets, huggingface_hub
# Installed but used internally: bitsandbytes (quantization), accelerate (device management)
!pip install "transformers>=4.44,<5.0" bitsandbytes accelerate peft trl datasets huggingface_hub

In [2]:
# CELL 2: Authenticate HuggingFace Hub
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret('HF_TOKEN')

if hf_token:
    login(token=hf_token)

In [3]:
# CELL 3: GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
Device: Tesla T4
VRAM: 15.6 GB


In [4]:
# CELL 4: Checkpoint Directory
import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
print("Checkpoint directory ready.")

Checkpoint directory ready.


In [5]:
# CELL 5: Load Dataset
from datasets import load_dataset

dataset = load_dataset("nicholas-ugbala-hf/medical-qa-narrative-10k")
train_dataset = dataset['train']
eval_dataset = dataset['eval']

print(f"Train: {len(train_dataset)}")
print(f"Eval: {len(eval_dataset)}")
print(f"Fields: {train_dataset[0].keys()}")

Train: 8220
Eval: 914
Fields: dict_keys(['instruction', 'input', 'output', 'text'])


In [6]:
import transformers
import torch
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")

transformers: 4.57.6
torch: 2.10.0+cu128


In [7]:
# CELL 6: Load Model with built-in pad token

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Use Llama's built-in reserved pad token. No add_special_tokens, no resize.
tokenizer.pad_token = "<|finetune_right_pad_id|>"
tokenizer.padding_side = "right"

print("Model loaded.")
print(f"Pad token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")

Loading model...


2026-06-13 07:34:14.645293: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781336054.667390     222 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781336054.674756     222 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781336054.694339     222 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781336054.694356     222 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781336054.694359     222 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.
Pad token: <|finetune_right_pad_id|> (id: 128004)


In [8]:
# CELL 7: Configure LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['up_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [9]:
# CELL 8: Train

from trl import SFTTrainer, SFTConfig

from trl import SFTTrainer, SFTConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

training_args = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    dataset_text_field="text",
    num_train_epochs=1,            # changed from 2
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    warmup_steps=100,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,                # changed from 150, more frequent since fewer total steps
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete.")

/tmp/ipykernel_222/3348836213.py:9: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128004}.


Starting training...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
100,2.484700,2.438548,2.493454,0.487289,389380.000000
200,2.400900,2.377057,2.357541,0.496501,776508.000000
300,2.358600,2.344368,2.317820,0.501083,1165908.000000
400,2.352900,2.323518,2.338604,0.504545,1555205.000000
500,2.328300,2.315524,2.325371,0.505435,1937691.000000


Training complete.


In [10]:
# CELL 9: Save training logs
import json

with open("/kaggle/working/training_log.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

print("Training log saved.")

Training log saved.


In [11]:
model.eval()

EOS_ID = tokenizer.convert_tokens_to_ids("<|eot_id|>")

def generate_response(prompt):
    messages = [
        {"role": "system", "content": "If you are a doctor, please answer the medical questions based on the patient's description."},
        {"role": "user", "content": prompt}
    ]
    encoded_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        padding=True
    ).to(model.device)

    outputs = model.generate(
        input_ids=encoded_inputs["input_ids"],
        attention_mask=encoded_inputs["attention_mask"],
        max_new_tokens=256,
        do_sample=False,
        repetition_penalty=1.3,
        eos_token_id=EOS_ID,
        pad_token_id=tokenizer.pad_token_id,
    )
    input_length = encoded_inputs["input_ids"].shape[-1]
    response = outputs[0][input_length:]
    return tokenizer.decode(response, skip_special_tokens=True)

In [14]:
# CELL 11: Compare against baseline(greedy, reproducible)
questions = [
    "What are the early symptoms of type 2 diabetes?",
    "What is the first-line treatment for hypertension in adults?",
    "What causes iron deficiency anemia?",
    "How is malaria diagnosed and treated?",
    "What are the warning signs of a heart attack?"
]

print("FINE-TUNED MODEL OUTPUTS (greedy decoding)")
print("=" * 60)

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate_response(q)}")
    print("=" * 60)

FINE-TUNED MODEL OUTPUTS (greedy decoding)
Q: What are the early symptoms of type 2 diabetes?
A: The earliest signs and symptoms that may indicate Type-2 Diabetes include:

1) Increased thirst (polydipsia)
2) Frequent urination or increased urine production 
3) Fatigue due to lack of energy from glucose metabolism.
4) Blurred vision - This is usually caused by high blood sugar levels affecting your eye pressure which can lead to glaucoma if left untreated for long periods.

5) Slow healing wounds - High Blood Sugar Levels cause damage to small vessels in body leading slow recovery time when wound heals slowly than normal person would heal it faster with proper care & nutrition.
Q: What is the first-line treatment for hypertension in adults?
A: The initial management of high blood pressure (hypertension) involves lifestyle modifications and medications to lower blood pressure levels effectively without causing any adverse effects or side reactions.

Medications commonly used as an initi

In [13]:
model.push_to_hub("nicholas-ugbala-hf/llama-3.2-3b-medical-finetuned-v2")
tokenizer.push_to_hub("nicholas-ugbala-hf/llama-3.2-3b-medical-finetuned-v2")
print("Pushed: huggingface.co/nicholas-ugbala-hf/llama-3.2-3b-medical-finetuned-v2")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed: huggingface.co/nicholas-ugbala-hf/llama-3.2-3b-medical-finetuned-v2
